# Sensor Coregistration

For proper computation, the sensors (electrode channels or magnetic coils) must be aligned to the head model. <br>
Alignment steps:
1. Visualize initial alignment
2. Align fiducials (Aligns reference points, serves to align roughly and fast before refining)
3. Refine alignment using an ICP algorithm
4. Project points to head (🚨 I would only use this step if the shape of the EEG cap does not match the shape of the head. This can happend when either the EEG montage or Head model was taken from a template. I do this in simulations where the true location of electrodes is arbitrary anyways 🚨)
5. Save the transformation file. (Save even if you don't apply transformations. This will save an identity transformation.)

Notes: There is no batch wise run for this section as aligning electrodes can be delicate. Please visualise steps.

# Step Wise Run

Run the code step by step for a single subject.

## 📦 Imports

In [1]:
import mindmap as mm 
import mne 
import pyvista as pv
from pathlib import Path

## 🧠 Data

📥 **<font color=pink>Inputs</font>**
- model (BEM or FEM)
- `eeg`: `Raw`, `Epochs` or `Evoked` EEG data.

🛠️ **<font color=violet>Configuration</font>**:
- `fconfig`: Configuration file.
- `sub`: Subject id (position in list SUBS)

In [2]:
########### Parameters ############
fconfig = "../configurations/spike.yml"
sub     = 17
###################################

reader = mm.io.dataset_reader(fconfig)
fs_dir = reader.config["FS_DIR"]
mm_dir = reader.config["MM_DIR"]

data = reader(sub, "eeg-raw", "hmodel")


DATA SUMMARY
Dataset: spike
Subject: sub-pat17
Retrieved objects:
- eeg-raw
- hmodel
Missing objects:


## Sensor Coregistration 
🌟 **Visualize alignmnent**: after each registration step, come back to this plot to see the changes

In [3]:
def get_fiducials(obj):
    """
    Get fiducial position from MNE object. 

    Parameters
    ----------
    obj: mne.Raw | mne.Evoked | mne.Epochs
        MNE object. 

    Returns
    -------
    fid: np.ndarray (3,3)
        Fiducial position in [m]
    """
    montage = obj.info["dig"]

    fiducials = []
    for dig in montage:
        if dig["kind"]._name == "FIFFV_POINT_CARDINAL":
            fiducials.append(dig["r"])

    return fiducials

# outer skin surface
head = data["hmodel"]["surfs"][0]
head = mm.utils.surf_to_polydata(head["rr"], head["tris"])

# coregistration object (computes MRI fiducials from fsaverage)        
coreg   = mne.coreg.Coregistration(data["eeg-raw"].info, data["sub"], fs_dir, fiducials="estimated")
fid_mri = pv.PolyData([coreg.fiducials.get_positions()[p] for p in ["nasion", "lpa", "rpa"]])

# sensor positions
sens      = pv.PolyData(data["eeg-raw"]._get_channel_positions())
fid_sens  = pv.PolyData(get_fiducials(data["eeg-raw"]))

# plot
plotter = pv.Plotter()
plotter.add_mesh(head, label="head", smooth_shading=False, color="#D2A18C")
plotter.add_mesh(fid_mri, render_points_as_spheres=True, color="red", label="MRI fiducials", point_size=10)
plotter.add_mesh(sens, render_points_as_spheres=True, color="black", label="sensors", point_size=8)
plotter.add_mesh(fid_sens, render_points_as_spheres=True, color="purple", label="sensor fiducials", point_size=10)

plotter.add_legend(face="circle")
plotter.show()

    Triangle neighbors and vertex normals...
Using low resolution head model in /media/nino/DATA/SPIKE/derivatives/freesurfer-7.4.1/sub-pat17/bem/outer_skin.surf
    Triangle neighbors and vertex normals...
Estimating fiducials from fsaverage.
Nearest-neighbor searches will be significantly faster if scikit-learn is installed.
Nearest-neighbor searches will be significantly faster if scikit-learn is installed.
Nearest-neighbor searches will be significantly faster if scikit-learn is installed.
Nearest-neighbor searches will be significantly faster if scikit-learn is installed.


Widget(value='<iframe src="http://localhost:45051/index.html?ui=P_0x7b4b453ab710_0&reconnect=auto" class="pyvi…

#### ⚙️ **<font color=green>Fit fiducials</font>**

Changes are updated in the visualization above.

In [5]:
# fit fiducials
coreg.fit_fiducials(verbose="warning")

# plot
sens.points     = mne.transforms.apply_trans(coreg.trans, data["eeg"]._get_channel_positions())
fid_sens.points = mne.transforms.apply_trans(coreg.trans, get_fiducials(data["eeg"]))

#### ⚙️ **<font color=green>Refine with Iteratice Closest Point (ICP)</font>**
Changes are updated in the visualization above.

🛠️ **<font color=violet>ICP</font>**:
- `iterations`: Number of ICP iterations.
- `nasion_weight`: ???

In [124]:
################# Parameters ###############
iterations = 30
nasion_weight = 1
#############################################

# icp
coreg.fit_icp(n_iterations=iterations, nasion_weight=nasion_weight, verbose="warning")

# plot
sens.points     = mne.transforms.apply_trans(coreg.trans, data["eeg"]._get_channel_positions())
fid_sens.points = mne.transforms.apply_trans(coreg.trans, get_fiducials(data["eeg"]))

#### ⚙️ **<font color=green>Project to surface</font>**
Only recommended for fitting template sensors to MRI in simulations. The head model has low resolution so the vertices are quite far from each other for accurately projecting to the head.


In [7]:
if data["name"] == "simf":
    points = mm.utils.project_points_to_surface(mne.transforms.apply_trans(coreg.trans, data["eeg"]._get_channel_positions()), head)
    sens.points = points

    # create eeg with new points
    channels = dict(zip(data["eeg"].info["ch_names"], points))
    montage  = mne.channels.make_dig_montage(ch_pos=channels, coord_frame="head")
    eeg      = data["eeg"].set_montage(montage)

💾 **Save trans file** <br>
If you don't apply any transformation, still save the coreg.trans. It will save an identity transformation.

🛠️ **<font color=violet>Options</font>**:
- `overwrite`: Overwrite existing files if True

In [5]:
######### Parameters ############
overwrite = True 
#################################

if data["name"] == "spike":
    ftrans = mm_dir/data["sub"]/"eeg"/f"{data['sub']}_eeg-trans.fif"

    if not ftrans.parent.exists():
        ftrans.parent.mkdir(parents=True)

    mne.write_trans(ftrans, coreg.trans, overwrite=overwrite)
    print(f"\033[92mSuccessfully saved EEG transformation for subject {data['sub']}\033[0m")

elif data["name"] == "simf":
    feeg = mm_dir/data["sub"]/"eeg"/f"{data['sub']}_coreg-ave.fif"

    if not feeg.parent.exists():
        feeg.parent.mkdir(parents=True)

    eeg.save(feeg, overwrite=overwrite, verbose="warning")
    print(f"\033[92mSuccessfully saved EEG with new channel positions for subject {data['sub']}\033[0m")

Overwriting existing file.
Successfully saved EEG transformation for subject sub-pat17
